Building Structure Analysis

Calculates building structure metrics for each polygon (NHDA + RA),
then cross-joins by nhda_id so every row has:
  nhda_<metric>  = NHDA partner's value
  ra_<metric>    = RA partner's value

From LoD2_2025_classified.gpkg:
  - built_up_ratio             : building footprint area / polygon area [%]
  - building_volume_density    : sum(footprint * height) / polygon area [m³/m²]
  - building_count             : number of buildings
  - building_density           : buildings per hectare
  - dominant_roof_type         : most common roof type (residential only)
  - dominant_roof_type_pct     : share of dominant roof type [%]
  - avg_building_height        : mean measured_height [m]
  - avg_building_footprint     : mean footprint area [m²]

From LoD2_2025_residential_types.gpkg:
  - dominant_res_subclass      : residential subclass with most buildings
  - dominant_res_subclass_pct  : share of that subclass among all buildings [%]
  - res_subclass_share_*       : share of each res_subclass category [%]


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ============================================================================
# HELPER
# ============================================================================

def get_buildings_within(polygon_geom, buildings_gdf):
    candidate_idx = list(buildings_gdf.sindex.intersection(polygon_geom.bounds))
    if not candidate_idx:
        return buildings_gdf.iloc[0:0]
    candidates = buildings_gdf.iloc[candidate_idx]
    return candidates[candidates.intersects(polygon_geom)]


def sanitize_subclass_name(name):
    txt = str(name).strip().lower()
    out = []
    for ch in txt:
        if ch.isalnum():
            out.append(ch)
        else:
            out.append('_')
    clean = ''.join(out)
    while '__' in clean:
        clean = clean.replace('__', '_')
    return clean.strip('_') or 'unknown'


def build_res_subclass_colmap(res_subclasses):
    col_map = {}
    used = set()
    for subclass in res_subclasses:
        base = f"res_subclass_share_{sanitize_subclass_name(subclass)}"
        col = base
        i = 2
        while col in used:
            col = f"{base}_{i}"
            i += 1
        used.add(col)
        col_map[str(subclass)] = col
    return col_map


# ============================================================================
# METRIC CALCULATIONS
# ============================================================================

def calculate_built_up_metrics(polygon_geom, buildings_gdf):
    polygon_area = polygon_geom.area
    polygon_area_ha = polygon_area / 10_000
    result = {
        'built_up_ratio': 0.0,
        'building_volume_density': 0.0,
        'building_count': 0,
        'building_density': 0.0,
        'avg_building_height': 0.0,
        'avg_building_footprint': 0.0,
    }
    if polygon_area <= 0:
        return result
    within = get_buildings_within(polygon_geom, buildings_gdf)
    if within.empty:
        return result

    footprint_areas = within.geometry.area
    total_footprint = footprint_areas.sum()
    result['built_up_ratio'] = round((total_footprint / polygon_area) * 100, 2)
    result['building_count'] = len(within)
    result['building_density'] = round(len(within) / polygon_area_ha, 2)

    if 'measured_height' in within.columns:
        heights = within['measured_height'].fillna(0)
        total_volume = (footprint_areas * heights).sum()
        result['building_volume_density'] = round(total_volume / polygon_area, 3)

        heights_no_na = within['measured_height'].dropna()
        if len(heights_no_na) > 0:
            result['avg_building_height'] = round(heights_no_na.mean(), 2)

    areas = within.geometry.area
    valid_areas = areas[areas > 0]
    if len(valid_areas) > 0:
        result['avg_building_footprint'] = round(valid_areas.mean(), 2)
    return result


def calculate_roof_type(polygon_geom, buildings_gdf):
    result = {'dominant_roof_type': None, 'dominant_roof_type_pct': None}
    within = get_buildings_within(polygon_geom, buildings_gdf)
    if within.empty:
        return result
    if 'building_class' not in within.columns or 'roof_type' not in within.columns:
        return result
    residential = within[within['building_class'] == 'residential']
    if residential.empty:
        return result
    roof_counts = residential['roof_type'].dropna().value_counts()
    if len(roof_counts) > 0:
        result['dominant_roof_type'] = roof_counts.index[0]
        result['dominant_roof_type_pct'] = round((roof_counts.iloc[0] / len(residential)) * 100, 1)
    return result


def calculate_residential_metrics(polygon_geom, res_buildings_gdf, subclass_col_map):
    result = {
        'dominant_res_subclass': None,
        'dominant_res_subclass_pct': None,
    }
    for col in subclass_col_map.values():
        result[col] = 0.0

    within = get_buildings_within(polygon_geom, res_buildings_gdf)
    if within.empty:
        return result

    if 'res_subclass' not in within.columns:
        return result

    # Shares are based on building counts inside polygon.
    # No age filter is applied, so "new buildings" are included if present in input data.
    total_buildings = len(within)
    if total_buildings == 0:
        return result

    subclass_counts = within['res_subclass'].dropna().astype(str).value_counts()
    if len(subclass_counts) == 0:
        return result

    dominant_subclass = subclass_counts.index[0]
    result['dominant_res_subclass'] = dominant_subclass
    result['dominant_res_subclass_pct'] = round((subclass_counts.iloc[0] / total_buildings) * 100, 1)

    for subclass_name, out_col in subclass_col_map.items():
        cnt = int(subclass_counts.get(subclass_name, 0))
        result[out_col] = round((cnt / total_buildings) * 100, 2)

    return result


# ============================================================================
# MAIN PROCESSING FUNCTION
# ============================================================================

BASE_METRIC_COLS = [
    'built_up_ratio',
    'building_volume_density',
    'building_count',
    'building_density',
    'dominant_roof_type',
    'dominant_roof_type_pct',
    'dominant_res_subclass',
    'dominant_res_subclass_pct',
    'avg_building_height',
    'avg_building_footprint',
]


def add_building_structure(
    input_file,
    lod2_classified_file,
    lod2_residential_file,
    output_file,
    dataset_name="Dataset",
    verbose=True,
    test_rows=None,
):
    print(f"\n{'='*80}")
    print(f"BUILDING STRUCTURE ANALYSIS – {dataset_name.upper()}")
    print(f"{'='*80}")

    # ------------------------------------------------------------------
    # Load
    # ------------------------------------------------------------------
    print("Loading input file...")
    gdf = gpd.read_file(input_file)
    if test_rows is not None:
        gdf = gdf.iloc[:test_rows].copy()
        print(f"  TEST MODE: using first {test_rows} rows only")
    print(f"  -> {len(gdf)} features, {gdf['nhda_id'].nunique()} unique nhda_ids")
    print(f"  -> type values: {gdf['type'].unique()}")
    print(f"  -> CRS: {gdf.crs}")

    print("\nLoading LoD2 classified buildings...")
    lod2 = gpd.read_file(lod2_classified_file)
    print(f"  -> {len(lod2)} buildings")

    print("\nLoading LoD2 residential buildings...")
    lod2_res = gpd.read_file(lod2_residential_file)
    print(f"  -> {len(lod2_res)} buildings")

    # ------------------------------------------------------------------
    # CRS alignment
    # ------------------------------------------------------------------
    if lod2.crs != gdf.crs:
        lod2 = lod2.to_crs(gdf.crs)
    if lod2_res.crs != gdf.crs:
        lod2_res = lod2_res.to_crs(gdf.crs)
    _ = lod2.sindex
    _ = lod2_res.sindex
    print("\n  ✓ CRS aligned, spatial indices built")

    # ------------------------------------------------------------------
    # Dynamic res_subclass share columns
    # ------------------------------------------------------------------
    if 'res_subclass' in lod2_res.columns:
        res_subclasses = sorted(lod2_res['res_subclass'].dropna().astype(str).unique().tolist())
    else:
        res_subclasses = []

    subclass_col_map = build_res_subclass_colmap(res_subclasses)
    subclass_share_cols = list(subclass_col_map.values())
    metric_cols = BASE_METRIC_COLS + subclass_share_cols

    if subclass_share_cols:
        print(f"\nDetected {len(subclass_share_cols)} res_subclass categories for share columns:")
        for subclass_name, col_name in subclass_col_map.items():
            print(f"  - {subclass_name} -> {col_name}")
    else:
        print("\nNo res_subclass categories detected. No share columns will be added.")

    # ------------------------------------------------------------------
    # Step 1: Compute metrics for every row
    # ------------------------------------------------------------------
    for col in metric_cols:
        gdf[col] = None

    print(f"\nStep 1: Computing building structure for all {len(gdf)} rows...")
    for i, (idx, feature) in enumerate(gdf.iterrows()):
        if i % 100 == 0:
            print(f"  {i+1}/{len(gdf)}...")
        polygon_geom = feature.geometry

        for col, val in calculate_built_up_metrics(polygon_geom, lod2).items():
            gdf.at[idx, col] = val
        for col, val in calculate_roof_type(polygon_geom, lod2).items():
            gdf.at[idx, col] = val
        for col, val in calculate_residential_metrics(polygon_geom, lod2_res, subclass_col_map).items():
            gdf.at[idx, col] = val

    print(f"  ✓ Done — buildings found in {(gdf['building_count'] > 0).sum()} / {len(gdf)} rows")

    # ------------------------------------------------------------------
    # Step 2: Cross-join NHDA ↔ RA values by nhda_id
    # ------------------------------------------------------------------
    print("\nStep 2: Cross-joining NHDA ↔ RA values by nhda_id...")

    nhda_lookup = (
        gdf[gdf['type'] == 'NHDA'][['nhda_id'] + metric_cols]
        .rename(columns={c: f'nhda_{c}' for c in metric_cols})
    )
    ra_lookup = (
        gdf[gdf['type'] == 'RA'][['nhda_id'] + metric_cols]
        .rename(columns={c: f'ra_{c}' for c in metric_cols})
    )

    gdf = gdf.merge(nhda_lookup, on='nhda_id', how='left')
    gdf = gdf.merge(ra_lookup, on='nhda_id', how='left')
    gdf = gdf.drop(columns=metric_cols)
    gdf = gpd.GeoDataFrame(gdf, crs=gpd.read_file(input_file).crs)

    print(f"  ✓ {len(gdf)} rows, nhda_* and ra_* columns added")

    # ------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------
    print(f"\n{'='*80}")
    print("SUMMARY")
    print(f"{'='*80}")
    for type_val, prefix in [('NHDA', 'nhda'), ('RA', 'ra')]:
        subset = gdf[gdf['type'] == type_val]
        print(f"\n{type_val} rows ({len(subset)}):")
        for col in ['built_up_ratio', 'building_count', 'avg_building_height']:
            vals = subset[f'{prefix}_{col}'].dropna().astype(float)
            if len(vals) > 0:
                print(f"  {col}: mean={vals.mean():.2f}, median={vals.median():.2f}")

    # ------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_file, driver='GPKG')
    print(f"\n✓ Saved: {output_file}  ({output_path.stat().st_size / 1024 / 1024:.1f} MB)")

    return gdf


# ============================================================================
# MAIN
# ============================================================================

def main():

    INPUT_FILE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC.gpkg"
    OUTPUT_FILE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure.gpkg"

    LOD2_CLASSIFIED = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\LoD2\LoD2_2025_classified.gpkg"
    LOD2_RESIDENTIAL = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\LoD2\LoD2_2025_residential_types_thr_h11m.gpkg"

    TEST_ROWS = None   # Set to a number (e.g. 100) to test on a subset

    print("\n" + "="*80)
    print("BUILDING STRUCTURE ANALYSIS")
    print("="*80)

    all_ok = True
    for label, path in [
        ("Input file", INPUT_FILE),
        ("LoD2 classified", LOD2_CLASSIFIED),
        ("LoD2 residential", LOD2_RESIDENTIAL),
    ]:
        exists = Path(path).exists()
        print(f"  {'✓' if exists else 'X'} {label}: {Path(path).name}")
        if not exists:
            all_ok = False

    if not all_ok:
        print("\nMissing input files - aborting.")
        return

    start_time = datetime.now()

    result = add_building_structure(
        input_file=INPUT_FILE,
        lod2_classified_file=LOD2_CLASSIFIED,
        lod2_residential_file=LOD2_RESIDENTIAL,
        output_file=OUTPUT_FILE,
        dataset_name="Comparison LST NDVI (NHDA + RA)",
        verbose=False,
        test_rows=TEST_ROWS,
    )

    print(f"\n✓ Done in {datetime.now() - start_time}")
    print("="*80 + "\n")


if __name__ == "__main__":
    main()